# 数据预处理

本文按八个阶段记录现有附件的检查、处理规则、运行结果和文件交付。全部预处理代码均在本 Notebook 中；原附件保留，不覆盖已有数据。

**已完成的结果：** 250 个样本按 `beer` 对齐；73 个化学特征缺失值按训练中位数填补；采用训练参数生成标准化副本；论坛副本排除 3 个全空列；相关矩阵字段全部对应，来源核验与最终检查通过。

| 阶段 | 内容 | 主要结果 |
| --- | --- | --- |
| 1 | 读取与初步检查 | 三张样本表各 250 行，编号无重复 |
| 2 | 缺失结构与字段说明 | 论坛 3 列全空；231 个化学特征均有字段说明 |
| 3 | 化学数值表示核对 | 70 列可对照，保留 81 个差异记录 |
| 4 | 感官字段与评分标尺 | 保留 50 个连续属性；无法还原原始 0–6 分 |
| 5 | 样本对齐与结构副本 | 保留原缺失、连续感官值及处理记录 |
| 6 | 样本划分与缺失填补 | 训练/验证为 177/73；只填补原缺失 |
| 7 | 化学特征标准化 | 训练参数拟合，标准化可逆 |
| 8 | 矩阵核验与最终交付 | 相关矩阵核验通过，汇总编号 1–8 的文件 |

最终文件统一放在 `处理结果/最终数据`，中间数据、参数和历史版本归入 `处理结果/处理记录`。目录使用清楚的名称，重复运行时采用“第 2 版”等编号，避免覆盖。历史记录的搬迁位置见 `处理结果/处理记录/目录说明.md`。

复现时从上到下运行。本文仅完成基础预处理，未开展赛题求解。原始等级评分、部分浓度含义和待核查差异的限制在文末统一说明。


## 1．读取数据与初步检查

### 1.1 读取三张样本表

使用 `Path` 管理路径，用 `pandas` 读取附件 1、4、7。`tables` 以文件名保存各表；`metadata` 指定样本编号、条码和风格三个非测量字段。

`beer_id` 按文本读取，避免继续发生数字精度问题；源文件已经丢失的条码精度无法据此恢复。样本匹配使用 `beer`，不使用条码或原始行号。

移动项目后只需调整 `folder`。运行依赖为 `pandas`、`numpy`、`openpyxl`；缺包时在当前内核安装后重新运行。


In [89]:
from pathlib import Path
import pandas as pd

folder = Path(r"D:\黄洛霖的文件\竞赛\数模赛\MathorCup2026A\附件")
files = ["1.csv", "4.csv", "7.csv"]
metadata = ["beer", "beer_id", "tasting_category_fine"]

tables = {}
for name in files:
    tables[name] = pd.read_csv(
        folder / name,
        dtype={"beer_id": "string"},
        encoding="utf-8-sig"
    )

化合物表前 5 行用于检查列名和读取形式。总列数为 234，其中 3 列为样本信息，231 列为化学、理化或汇总特征。

预览中的 `beer` 为 10、86、2、83、40，左侧 0–4 是行索引，二者不能混用。前几行同属一个风格不能代表整体风格分布；省略号表示显示折叠，不表示缺失。


In [90]:
tables["1.csv"].head()

,beer,beer_id,tasting_category_fine,acetaldehyde,CS2,DES,DMS,ethyl_2.methyl_butyrate,ethyl_acetate,ethyl_decanoate,...,X13.epi.manool.oxide,isopropyl.hexadecanoate,manool,ethyl.octadecanoate,ethyl.pentanoate,X2.ethyl.3.methylpyrazine,esters_sum,aroma_hops_sum,acids_sum,sulfur_sum
0,10,5410228202929,Low/No alcohol,-0.641741,-2.146292,-1.905267,0.046211,-3.19552,-0.599350,-1.975309,...,4.360243,-1.817819,5.721204,4.438951,0.014728,0.054353,-0.532634,3.696586,0.244085,0.166286
1,86,5410783031019,Low/No alcohol,-0.174824,-2.146292,-0.257098,0.347241,-3.19552,-0.507254,-1.755970,...,4.512589,4.323292,5.810501,4.247826,0.023167,0.001400,-0.465401,1.219563,-0.152156,0.098931
2,2,5410228142003,Low/No alcohol,0.511067,-2.146292,-1.905267,0.898604,-3.19552,-0.134514,-1.893639,...,4.394267,-1.817819,5.585529,3.598293,0.017571,0.000699,-0.409346,1.148974,-0.161504,-0.022943
3,83,54055520,Low/No alcohol,0.249793,-2.146292,-1.905267,0.782126,-3.19552,-0.233874,-2.086981,...,4.174304,4.088807,5.204583,4.498781,0.016959,0.001101,-0.420080,1.416847,-0.056092,0.068372
4,40,5410228141181,Low/No alcohol,-1.902829,-0.756987,-1.347270,0.553356,-3.19552,0.207751,-1.713095,...,4.742899,4.651536,6.515499,4.789220,0.117921,0.000492,-0.144457,0.672328,-1.204001,0.639462


### 1.2 检查样本编号

验证三张表均有需要的样本信息；`beer` 无空值、无重复，且三张表的编号集合完全相同。行顺序不同不影响编号集合检查。

`assert` 表示必须满足的条件。检查失败时先定位原因，不跳过断言继续处理。


In [91]:
for name, df in tables.items():
    assert set(metadata).issubset(df.columns), f"{name} 缺少编号或风格列"
    assert df["beer"].notna().all(), f"{name} 的 beer 存在空值"
    assert df["beer"].is_unique, f"{name} 的 beer 存在重复"

assert (
    set(tables["1.csv"]["beer"])
    == set(tables["4.csv"]["beer"])
    == set(tables["7.csv"]["beer"])
), "三张表的样本编号不一致"

print("检查通过：beer 无空值、无重复，三张表包含相同的样本编号。")

检查通过：beer 无空值、无重复，三张表包含相同的样本编号。


**结果：** 三张表各有 250 个样本，`beer` 无空值、无重复，编号集合相同，可以按编号对应。风格标签的一致性在第 2 阶段单独核对。


### 1.3 统计规模与缺失总量

排除三个样本信息字段，统计各表的测量列数和缺失单元格数。`isna().sum()` 统计空值，空值数量不等于受影响样本数量。


In [92]:
records = []
for name, df in tables.items():
    records.append({
        "文件": name,
        "样本数": len(df),
        "测量列数": len(df.columns.difference(metadata)),
        "缺失单元格": int(df.isna().sum().sum())
    })

report = pd.DataFrame(records)
report

,文件,样本数,测量列数,缺失单元格
0,1.csv,250,231,73
1,4.csv,250,50,0
2,7.csv,250,186,750


**实际结果：**

| 原附件 | 样本数 | 测量列数 | 缺失单元格 |
| --- | ---: | ---: | ---: |
| 1.csv | 250 | 231 | 73 |
| 4.csv | 250 | 50 | 0 |
| 7.csv | 250 | 186 | 750 |

总列数分别为 234、53、189。专业感官表无空值不代表它提供了原始 0–6 分；论坛的 750 个缺失集中在第 2 阶段核对的 3 个全空列。


### 1.4 定位化学特征缺失

逐列计算缺失数量和缺失率。缺失率为“该列缺失数 ÷ 样本数”；`isna().mean()` 得到的小数 0.108 表示 10.8%。


In [93]:
chem = tables["1.csv"].drop(columns=metadata)

missing = pd.DataFrame({
    "缺失数量": chem.isna().sum(),
    "缺失率": chem.isna().mean()
})

missing_report = (
    missing.loc[missing["缺失数量"] > 0]
    .sort_values("缺失数量", ascending=False)
)
missing_report

,缺失数量,缺失率
amyl.acetate,27,0.108
aroma_hops_sum,13,0.052
methylpyrazine,12,0.048
X2.3.butanedione,9,0.036
X3.methylbutanal,7,0.028
ethyl.nicotinate,2,0.008
ethyl.isobutyrate,1,0.004
furfural,1,0.004
phenylacetaldehyde,1,0.004


**结果：** 9 个特征共缺失 73 个单元格，占 250×231 个单元格的约 0.1264%。

| 特征 | 缺失数量 | 缺失率 |
| --- | ---: | ---: |
| amyl.acetate | 27 | 10.8% |
| aroma_hops_sum | 13 | 5.2% |
| methylpyrazine | 12 | 4.8% |
| X2.3.butanedione | 9 | 3.6% |
| X3.methylbutanal | 7 | 2.8% |
| ethyl.nicotinate | 2 | 0.8% |
| ethyl.isobutyrate | 1 | 0.4% |
| furfural | 1 | 0.4% |
| phenylacetaldehyde | 1 | 0.4% |

这些统计只说明缺失位置与程度，不说明原因。先核对字段与数值表示，再在第 6 阶段填补；此处不把空值、零值和负值混为一类。


**本阶段结论：** 三张样本表均可按 `beer` 对应。化学表有少量分散缺失，论坛表有整列缺失，二者采用不同的处理规则。


## 2．缺失结构与字段说明

### 2.1 定位论坛全空列

逐列统计论坛属性的缺失数和缺失率。整列没有观测时，无法从该列计算有依据的均值或中位数。


In [94]:
forum_features = tables["7.csv"].drop(columns=metadata)
forum_missing = pd.DataFrame({
    "缺失数量": forum_features.isna().sum(),
    "缺失率": forum_features.isna().mean()
})
forum_missing = forum_missing.loc[forum_missing["缺失数量"] > 0]

assert int(forum_missing["缺失数量"].sum()) == int(forum_features.isna().sum().sum())
print("存在缺失的属性数：", len(forum_missing))
print("整列缺失的属性数：", int(forum_features.isna().all().sum()))
print("至少有一个属性缺失的样本数：", int(forum_features.isna().any(axis=1).sum()))
forum_missing.sort_values("缺失数量", ascending=False)

存在缺失的属性数： 3
整列缺失的属性数： 3
至少有一个属性缺失的样本数： 250


,缺失数量,缺失率
palate_acetaldehyde,250,1.0
palate_acetate,250,1.0
palate_dentist,250,1.0


**结果：** `palate_acetaldehyde`、`palate_acetate`、`palate_dentist` 三列均缺失 250 个值，合计 750 个。其余 183 列无空值。

处理决定：保留全部 250 个样本，只在论坛数据副本中排除这三列，并保存排除记录；不填成零，不改原附件。


### 2.2 核对风格标签

将三张表按 `beer` 对齐，比较 `tasting_category_fine`。编号相同的样本还应对应相同风格，不能只核对行数。


In [95]:
styles = pd.concat(
    {
        name: df.set_index("beer")["tasting_category_fine"]
        for name, df in tables.items()
    },
    axis=1
).sort_index()

assert styles.notna().all().all(), "存在缺失的风格标签，请先核对。"
style_conflicts = styles.loc[styles.nunique(axis=1) != 1]
print("风格不一致的样本数：", len(style_conflicts))
assert style_conflicts.empty, "同一 beer 的风格不一致，请查看 style_conflicts。"
styles.head()

风格不一致的样本数： 0


,1.csv,4.csv,7.csv
beer,,,
1,Brut,Brut,Brut
2,Low/No alcohol,Low/No alcohol,Low/No alcohol
3,Blond,Blond,Blond
4,Strong ale,Strong ale,Strong ale
5,Blond,Blond,Blond


**结果：** 三张表风格不一致的样本数为 0。后续统一按 `beer` 对应和排序。


### 2.3 建立化学特征字段表

读取附件 8 的 `Sensory thresholds`，以 `field_name_R` 对应主表字段，合并化学名称、校准标记和参考单位；同时记录缺失、范围、零值及负值数量。

所有 231 个主表特征都找到了字段说明。字段说明与逐样本的浓度参照需要分开核对。


In [96]:
reference = pd.read_excel(
    folder / "8.xlsx",
    sheet_name="Sensory thresholds"
).set_index("field_name_R")

assert reference.index.is_unique, "参考表字段名重复，不能直接一对一匹配。"
assert set(chem.columns).issubset(set(reference.index)), "部分特征未找到字段说明。"

field_dictionary = pd.DataFrame({
    "缺失数量": chem.isna().sum(),
    "缺失率": chem.isna().mean(),
    "最小值": chem.min(),
    "最大值": chem.max(),
    "负值数量": (chem < 0).sum(),
    "零值数量": (chem == 0).sum()
}).rename_axis("field_name_R")

field_dictionary = field_dictionary.join(
    reference[["chemical_name", "calibrated", "unit"]],
    how="left",
    validate="one_to_one"
)

assert len(field_dictionary) == chem.shape[1]
print("已整理的特征数：", len(field_dictionary))
field_dictionary.head(12)

已整理的特征数： 231


,缺失数量,缺失率,最小值,最大值,负值数量,零值数量,chemical_name,calibrated,unit
field_name_R,,,,,,,,,
acetaldehyde,0,0.0,-1.902829,1.268628,11,0,Acetaldehyde,yes,mg/L
CS2,0,0.0,-2.146292,-0.008552,250,0,Carbon disulfide,yes,ug/L
DES,0,0.0,-1.905267,0.671647,181,0,Diethyl sulfide,yes,ug/L
DMS,0,0.0,0.046211,2.716380,0,0,Dimethyl sulfide,yes,ug/L
ethyl_2.methyl_butyrate,0,0.0,-3.195520,-0.539583,250,0,Ethyl 2-methylbutyrate,yes_gcms,mg/L
ethyl_acetate,0,0.0,-0.599350,2.116964,4,0,Ethyl acetate,yes,mg/L
ethyl_decanoate,0,0.0,-2.975720,-0.186983,250,0,Ethyl decanoate,"yes_gcms, yes",mg/L
ethyl_hexanoate,0,0.0,-2.476904,-0.078259,250,0,Ethyl hexanoate,"yes_gcms,yes",mg/L
ethyl_isovalerate,0,0.0,-4.487004,-0.858848,250,0,Ethyl 3-methylbutyrate,"yes_gcms,yes",mg/L


**解释边界：** 参考表中的 `unit` 和 `calibrated` 不能证明主表当前值就是原始物理浓度。部分列有负值，需核对对数等表示形式；单位缺失时不自行补单位。下一阶段使用逐样本参考值检查这些关系。


## 3．核对化学数值表示

### 3.1 对齐主表与参考浓度

读取 `Calibrated chemical dataset`，按样本编号和特征名对齐。两表共有 70 个测量指标，其余 161 个特征没有直接浓度参照。

**检查要求：** 样本与风格一致，仅对共有指标判断；不把部分列的关系推广到全部特征。


In [97]:
import numpy as np

chemical_values = (
    tables["1.csv"].set_index("beer")
    .drop(columns=["beer_id", "tasting_category_fine"])
)
calibrated_table = pd.read_excel(
    folder / "8.xlsx",
    sheet_name="Calibrated chemical dataset"
).set_index("beer")

assert calibrated_table.index.notna().all() and calibrated_table.index.is_unique
assert set(calibrated_table.index) == set(chemical_values.index), "参考表的样本编号不一致。"
assert calibrated_table["tasting_category_fine"].reindex(chemical_values.index).equals(
    tables["1.csv"].set_index("beer")["tasting_category_fine"]
), "参考表与主表的风格不一致。"

common_features = [c for c in chemical_values.columns if c in calibrated_table.columns]
no_reference_features = [c for c in chemical_values.columns if c not in calibrated_table.columns]
reference_values = calibrated_table.loc[chemical_values.index, common_features]

assert len(common_features) > 0, "没有可直接对照的指标。"
print("已对齐的样本数：", len(chemical_values))
print("可直接对照的指标数：", len(common_features))
print("暂时没有对应参考浓度的指标数：", len(no_reference_features))

已对齐的样本数： 250
可直接对照的指标数： 70
暂时没有对应参考浓度的指标数： 161


### 3.2 理解原值与对数关系

设主表值为 x、参考值为 c，比较 `x≈c`、`x≈log10(c)`、`x≈ln(c)`。先查看 `beer=10` 的四个指标，只计算关系，不改主表。


In [98]:
example_beer = 10
example_features = ["pH", "ethanol..v.v.", "acetaldehyde", "diethyl.succinate"]

examples = pd.DataFrame({
    "主表值": chemical_values.loc[example_beer, example_features],
    "参考原值": reference_values.loc[example_beer, example_features]
})
positive_reference = examples["参考原值"].where(examples["参考原值"] > 0)
examples["参考值的log10"] = np.log10(positive_reference)
examples["参考值的ln"] = np.log(positive_reference)
examples

,主表值,参考原值,参考值的log10,参考值的ln
pH,0.624798,4.215000,0.624798,1.438650
ethanol..v.v.,-1.154902,0.070000,-1.154902,-2.659260
acetaldehyde,-0.641741,0.228170,-0.641741,-1.477664
diethyl.succinate,0.005678,0.005678,-2.245793,-5.171130


**示例结果：** pH、酒精含量和 acetaldehyde 与参考值符合 log10 关系；diethyl.succinate 更接近原值关系。

不同列可能使用不同表示，不能把所有负值视为错误，也不能把整表再取一次对数。单一样本只用于说明公式，整列结果以下面的核验为准。


### 3.3 批量检查共有指标

仅使用两表值均有限且参考值大于 0 的记录，比较三种关系，以绝对差不超过 10⁻⁶ 记为吻合。吻合数最多且唯一的关系记为候选，再区分全部吻合与仍有差异。

该容差用于数值核对，不是化学阈值或删值标准。参考零值、缺失值不参与对数比较；不擅自加常数。关系不明确时断言停止。


In [99]:
tolerance = 1e-6
comparison_records = []

for feature in common_features:
    main = chemical_values[feature]
    raw = reference_values[feature]
    usable = np.isfinite(main) & np.isfinite(raw) & (raw > 0)
    observed = main.loc[usable]
    raw_positive = raw.loc[usable]
    assert raw_positive.nunique() >= 2, f"{feature} 的可比正值不足，请单独核对。"

    candidates = {
        "原值": raw_positive,
        "log10": np.log10(raw_positive),
        "ln": np.log(raw_positive)
    }
    match_counts = {
        relation: int((abs(observed - expected) <= tolerance).sum())
        for relation, expected in candidates.items()
    }
    best_count = max(match_counts.values())
    winners = [relation for relation, count in match_counts.items() if count == best_count]
    assert best_count > 0 and len(winners) == 1, f"{feature} 无法区分关系，请单独核对。"
    candidate = winners[0]
    difference_count = len(observed) - best_count

    comparison_records.append({
        "特征": feature,
        "候选关系": candidate,
        "可比正值数": len(observed),
        "吻合数": best_count,
        "差异数": difference_count,
        "吻合率": best_count / len(observed),
        "参考零值数": int(raw.eq(0).sum()),
        "参考缺失数": int(raw.isna().sum()),
        "主表缺失数": int(main.isna().sum()),
        "核验状态": "可比正值全部吻合" if difference_count == 0 else "可比正值仍有差异"
    })

transform_audit = pd.DataFrame(comparison_records).set_index("特征")
assert len(transform_audit) == len(common_features)
assert (transform_audit["吻合数"] + transform_audit["差异数"]).equals(transform_audit["可比正值数"])
transform_audit.groupby(["候选关系", "核验状态"]).size().rename("特征数").to_frame()

特征数
候选关系  核验状态         
log10 可比正值全部吻合   37
原值    可比正值仍有差异   32
      可比正值全部吻合    1

**结果：** 37 列的可比正值全部符合 log10 关系；33 列以原值为候选，其中 1 列全部吻合、32 列仍有差异。没有列以 ln 作为唯一最优候选。

该结果只覆盖 70 个共有指标，未覆盖的 161 列保留现状，不进行整表反变换。


### 3.4 保存差异与核验范围

按候选关系生成 `beer×特征` 的差异清单，保存双方值和差值；把核验状态合并回字段表。未覆盖列标注“无对应参考浓度”，不据此删除特征。


In [100]:
difference_tables = []
for feature in common_features:
    main = chemical_values[feature]
    raw = reference_values[feature]
    usable = np.isfinite(main) & np.isfinite(raw) & (raw > 0)
    observed = main.loc[usable]
    raw_positive = raw.loc[usable]
    relation = transform_audit.loc[feature, "候选关系"]
    expected = {
        "原值": raw_positive,
        "log10": np.log10(raw_positive),
        "ln": np.log(raw_positive)
    }[relation]

    detail = pd.DataFrame({
        "主表值": observed,
        "参考原值": raw_positive,
        "按候选关系计算的参考值": expected
    })
    detail = detail.loc[abs(observed - expected) > tolerance].copy()
    detail["特征"] = feature
    detail["候选关系"] = relation
    detail["差值"] = detail["主表值"] - detail["按候选关系计算的参考值"]
    difference_tables.append(detail.reset_index())

positive_differences = pd.concat(difference_tables, ignore_index=True)
assert len(positive_differences) == int(transform_audit["差异数"].sum())

field_dictionary_checked = field_dictionary.join(
    transform_audit[["候选关系", "核验状态"]],
    how="left",
    validate="one_to_one"
)
field_dictionary_checked["核验状态"] = field_dictionary_checked["核验状态"].fillna("无对应参考浓度")
assert len(field_dictionary_checked) == len(field_dictionary)

print("可比正值中的差异单元格数：", len(positive_differences))
print("没有对应参考浓度的特征数：", len(no_reference_features))
positive_differences.head(15)

可比正值中的差异单元格数： 81
没有对应参考浓度的特征数： 161


,beer,主表值,参考原值,按候选关系计算的参考值,特征,候选关系,差值
0,151,0.139752,0.043173,0.043173,X2.3.butanedione,原值,0.096579
1,156,0.434130,0.123114,0.123114,X2.3.butanedione,原值,0.311016
2,34,0.289622,0.530466,0.530466,X2.3.butanedione,原值,-0.240844
3,158,0.108457,0.371494,0.371494,X3.methylbutanal,原值,-0.263037
4,126,0.149200,0.303057,0.303057,ethyl.isobutyrate,原值,-0.153858
5,123,0.017725,0.796353,0.796353,ethyl.isobutyrate,原值,-0.778628
6,235,0.124520,0.600187,0.600187,ethyl.isobutyrate,原值,-0.475667
7,89,0.057612,0.904792,0.904792,ethyl.isobutyrate,原值,-0.847181
8,103,0.017725,0.403482,0.403482,ethyl.isobutyrate,原值,-0.385757
9,151,0.060652,0.589902,0.589902,X2.methyltetrahydrofuran.3.one,原值,-0.529251


**结果：** 差异清单有 81 行，不代表确认了 81 个错误值。例如 beer=156 的一项指标，两表约为 12596.432870 与 12596.432872，仅相差约 0.000002，也会被固定绝对容差标记。

原值和对数值的差异尺度不同，不混合排序判断严重程度。当前只保存标记，不使用参考值覆盖主表；该清单也不是完整异常值清单。


**本阶段结论：** 已记录主表可核实的数值表示及其限制。化学值整体保持现有表示，负值、零值和差异记录均未被自动裁剪或替换。


## 4．整理感官字段与评分标尺

### 4.1 建立感官属性清单

保留附件 4 全部 50 个属性，使用附件 8 的 `Sensory names` 建立说明，并区分 A_、F_ 和其他属性。

实际为 A_ 15 项、F_ 15 项、其他感官/总体 20 项。题面“约 40 项”不能作为凑数删列规则；参考表 `DataType` 也不代表现有连续值可直接作为等级标签。


In [101]:
sensory_values = (
    tables["4.csv"].set_index("beer")
    .drop(columns=["beer_id", "tasting_category_fine"])
)
sensory_names = pd.read_excel(folder / "8.xlsx", sheet_name="Sensory names")
trained_names = (
    sensory_names.loc[sensory_names["dataset"].eq("Trained panel")]
    .set_index("R_name")
)
assert trained_names.index.is_unique, "感官属性说明存在重复字段名。"
assert set(sensory_values.columns).issubset(set(trained_names.index)), "部分感官属性未找到说明。"

label_groups = pd.Series("其他感官/总体", index=sensory_values.columns, name="目标分组")
label_groups.loc[label_groups.index.str.startswith("A_")] = "Aroma（A_）"
label_groups.loc[label_groups.index.str.startswith("F_")] = "Flavour（F_）"

target_dictionary = label_groups.to_frame().join(
    trained_names[["Category_Paper", "CleanName_PaperLong", "DataType"]],
    how="left",
    validate="one_to_one"
).rename_axis("感官属性")

assert len(target_dictionary) == sensory_values.shape[1]
print("样本数：", len(sensory_values))
print("感官属性总数：", len(target_dictionary))
target_dictionary.groupby("目标分组").size().rename("属性数").to_frame()

样本数： 250
感官属性总数： 50


,属性数
目标分组,
Aroma（A_）,15
Flavour（F_）,15
其他感官/总体,20


### 4.2 检查原始 0–6 分的数值形式

原始等级应为 0–6 的整数，检查现有值中的非整数、负值及越界数量。题面明确附件 4 已标准化/中心化；这些统计用于核验数值形式，不能据此恢复原始分数。


In [102]:
assert np.isfinite(sensory_values.to_numpy()).all(), "感官数据存在缺失或无穷值，请先核对。"
non_integer = (sensory_values - sensory_values.round()).abs() > 1e-8
outside_score_range = (sensory_values < 0) | (sensory_values > 6)

sensory_scale_audit = pd.DataFrame({
    "最小值": sensory_values.min(),
    "最大值": sensory_values.max(),
    "负值数量": (sensory_values < 0).sum(),
    "非整数数量": non_integer.sum(),
    "超出0至6数量": outside_score_range.sum()
})
sensory_scale_audit["符合原始0至6数值形式"] = (
    sensory_scale_audit["非整数数量"].eq(0)
    & sensory_scale_audit["超出0至6数量"].eq(0)
)

print("不符合原始离散0–6数值形式的属性数：", int((~sensory_scale_audit["符合原始0至6数值形式"]).sum()))
print("全部现有连续值的范围：", float(sensory_values.min().min()), "至", float(sensory_values.max().max()))
sensory_scale_audit.loc[["A_malt_all", "F_malt_all", "acidity", "overall"]]

不符合原始离散0–6数值形式的属性数： 50
全部现有连续值的范围： -2.150728655 至 6.172383248


,最小值,最大值,负值数量,非整数数量,超出0至6数量,符合原始0至6数值形式
A_malt_all,-1.552115,2.876477,139,250,139,False
F_malt_all,-1.649828,2.466214,127,250,127,False
acidity,-0.725543,3.781348,192,250,192,False
overall,-1.735965,1.345650,116,250,116,False


**结果：** 50 个属性均不符合原始离散 0–6 分形式，连续值整体约为 -2.150729 至 6.172383。

保留连续值，不四舍五入、不裁剪、不做 MinMax 拉伸来冒充原始等级。用户已确认没有其他原始评分或转换参数，现有材料无法支持还原。


### 4.3 检查评分卡输入

只读检查附件 3 中评分公式依赖的输入和描述符勾选位置。空输入也可能使公式缓存显示零，这与人工给出零分不同。

`data_only=False` 读取公式与输入；`data_only=True` 读取已有缓存，不会自动重算。代码不保存评分卡。


In [103]:
from openpyxl import load_workbook

cards_formulas = load_workbook(folder / "3.xlsx", data_only=False)
cards_cached = load_workbook(folder / "3.xlsx", data_only=True)
score_rows = [7, 20, 33, 45, 50, 54, 58, 62, 68, 72, 76, 82, 86, 90, 94]
score_columns = list(range(3, 16, 2)) + list(range(19, 32, 2))
descriptor_rows = [10, 12, 14, 16, 23, 25, 27, 29, 36, 38, 40, 42]
card_records = []

for sheet in cards_formulas.sheetnames:
    if not sheet.startswith("beer_"):
        continue
    ws = cards_formulas[sheet]
    card_records.append({
        "工作表": sheet,
        "卡片Beer编号": ws["S3"].value,
        "评分输入非空数": sum(ws.cell(r, c).value is not None for r in score_rows for c in score_columns),
        "描述符勾选非空数": sum(ws.cell(r, c).value is not None for r in descriptor_rows for c in [3, 19]),
        "Q7公式缓存值": cards_cached[sheet]["Q7"].value
    })

card_summary_headers = [
    cell.value for cell in cards_formulas["summary"][1]
    if cell.value is not None and cell.value not in ["Beer", "comments"]
]
cards_formulas.close()
cards_cached.close()

card_audit = pd.DataFrame(card_records)
assert not card_audit.empty, "未找到评分卡工作表。"
print("实际评分卡工作表数：", len(card_audit))
print("评分输入非空总数：", int(card_audit["评分输入非空数"].sum()))
print("描述符勾选非空总数：", int(card_audit["描述符勾选非空数"].sum()))
card_audit

实际评分卡工作表数： 12
评分输入非空总数： 0
描述符勾选非空总数： 0


,工作表,卡片Beer编号,评分输入非空数,描述符勾选非空数,Q7公式缓存值
0,beer_1,1,0,0,0
1,beer_2,2,0,0,0
2,beer_3,3,0,0,0
3,beer_4,4,0,0,0
4,beer_5,5,0,0,0
5,beer_6,6,0,0,0
6,beer_7,7,0,0,0
7,beer_8,8,0,0,0
8,beer_9,9,0,0,0
9,beer_10,10,0,0,0


**结果：** 实际有 12 张评分卡，评分输入和描述符勾选的非空总数均为 0。Q7 的缓存零值不能作为真实评分。

附件 3 提供空白评分模板和字段定义。本次最终交付保留原样副本，不填入推断分数。


### 4.4 对应评分卡与连续感官字段

只建立三个拼写别名：`A_ester_fruity→A_esters_fruity`、`F_ester_fruity→F_esters_fruity`、`4vg→X4vg`。最终 50 个字段全部对应。

字段定义能对应不等于取得了原始评分。分别记录字段含义、连续值标尺和原始等级不可恢复的状态。


In [104]:
card_aliases = {
    "A_ester_fruity": "A_esters_fruity",
    "F_ester_fruity": "F_esters_fruity",
    "4vg": "X4vg"
}
card_schema = pd.DataFrame({"评分卡summary列名": card_summary_headers})
card_schema["对应4.csv列名"] = card_schema["评分卡summary列名"].replace(card_aliases)
assert card_schema["对应4.csv列名"].is_unique, "名称对应后出现重复。"
assert set(card_schema["对应4.csv列名"]) == set(sensory_values.columns), "两表字段尚未完全对应。"
assert card_audit[["评分输入非空数", "描述符勾选非空数"]].to_numpy().sum() == 0, "评分卡存在填写值，请先解析后再判断标尺。"

target_schema = target_dictionary.copy()
target_schema["标签来源"] = "4.csv"
target_schema["当前值标尺"] = "标准化/中心化连续值（题面说明）"
target_schema["评分卡summary定义存在"] = target_schema.index.isin(card_schema["对应4.csv列名"])
target_schema["原始0至6观测状态"] = "当前评分卡输入为空，尚未核实"
assert len(target_schema) == sensory_values.shape[1]

print("已对应的感官属性数：", len(target_schema))
print("当前仅整理观测与字段，不生成或推断新的感官分数。")
card_schema.loc[card_schema["评分卡summary列名"] != card_schema["对应4.csv列名"]]

已对应的感官属性数： 50
当前仅整理观测与字段，不生成或推断新的感官分数。


,评分卡summary列名,对应4.csv列名
14,A_ester_fruity,A_esters_fruity
29,F_ester_fruity,F_esters_fruity
39,4vg,X4vg


**本阶段结论：** 专业感官表为 250×50，现有数值全部保留。评分卡字段已对应，原始 0–6 分不可恢复；最终标尺说明保存在第 8 阶段的字段表中。


以下开始生成可追溯的数据副本。按 `beer` 对齐样本，保留化学原缺失和差异记录，排除论坛全空列；随后分别填补、标准化。原附件保持不变。


## 5．生成结构整理副本

### 5.1 按样本编号统一排序

生成样本信息、化学特征、专业感官及论坛属性四张表，保留全部 250 个样本。化学表的 73 个缺失、零值、负值与待核查差异仍保留。

导出记录统一放在 `处理结果/处理记录`。`next_output_path` 先使用阶段名称；同名目录存在时用“第 2 版”等可读编号，导出采用排他创建，防止覆盖。


In [105]:
import hashlib

source_hashes_before = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
}
output_root = folder.parent / "处理结果" / "处理记录"
def next_output_path(base):
    base = Path(base)
    candidate = base
    version = 2
    while candidate.exists():
        candidate = base.with_name(f"{base.name}（第{version}版）")
        version += 1
    return candidate

sample_info = (
    tables["1.csv"].set_index("beer")[["beer_id", "tasting_category_fine"]]
    .sort_index().copy()
)
chemical_aligned = chemical_values.reindex(sample_info.index).copy()
sensory_aligned = sensory_values.reindex(sample_info.index).copy()
forum_aligned = (
    tables["7.csv"].set_index("beer")
    .drop(columns=["beer_id", "tasting_category_fine"])
    .reindex(sample_info.index).copy()
)
empty_forum_columns = forum_aligned.columns[forum_aligned.isna().all()].tolist()
forum_no_empty = forum_aligned.drop(columns=empty_forum_columns).copy()

assert set(empty_forum_columns) == set(forum_missing.index[forum_missing["缺失率"].eq(1)])
print("统一样本数：", len(sample_info))
print("从论坛副本排除的全空属性：", empty_forum_columns)
print("保留的论坛属性数：", forum_no_empty.shape[1])


统一样本数： 250
从论坛副本排除的全空属性： ['palate_acetaldehyde', 'palate_acetate', 'palate_dentist']
保留的论坛属性数： 183


### 5.2 保存原缺失和排除列记录

`np.where` 找到原缺失的行列位置，再对应回 `beer` 与特征名；另存三列论坛属性的排除依据，便于区分原始观测与后续填补。


In [106]:
missing_row_positions, missing_column_positions = np.where(chemical_aligned.isna().to_numpy())
chemical_missing_positions = pd.DataFrame({
    "beer": chemical_aligned.index[missing_row_positions],
    "特征": chemical_aligned.columns[missing_column_positions]
})
removed_forum_columns = pd.DataFrame({
    "特征": empty_forum_columns,
    "原始缺失数量": [int(forum_aligned[c].isna().sum()) for c in empty_forum_columns],
    "处理": "仅从论坛属性副本排除",
    "依据": "全部样本均缺失，没有有效观测"
})

prepared_tables = {
    "样本信息": sample_info,
    "化合物_保留缺失": chemical_aligned,
    "专业感官_连续值": sensory_aligned,
    "论坛_去全空列": forum_no_empty
}
structure_summary = pd.DataFrame([
    {
        "数据表": name,
        "样本数": len(table),
        "字段数_不含beer": table.shape[1],
        "缺失单元格数": int(table.isna().sum().sum())
    }
    for name, table in prepared_tables.items()
])
assert len(chemical_missing_positions) == int(chemical_aligned.isna().sum().sum())
structure_summary

,数据表,样本数,字段数_不含beer,缺失单元格数
0,样本信息,250,2,0
1,化合物_保留缺失,250,231,73
2,专业感官_连续值,250,50,0
3,论坛_去全空列,250,183,0


**结构整理结果：** 样本信息为 250×2（原始文本条码与风格，beer 为索引）；化学表为 250×231，仍缺失 73 个值；专业感官为 250×50；论坛副本为 250×183。

条码按来源保留，其原有精度问题未被修复。化学特征与感官属性分别保存，后续通过 `beer` 对应。


### 5.3 检查副本与来源一致

验证样本、字段、顺序和保留数值一致，原缺失位置完整，没有新增无穷值；用文件校验值确认附件内容未改变。


In [107]:
for name, table in prepared_tables.items():
    assert table.index.is_unique and table.index.notna().all(), f"{name} 的样本索引有问题。"
    assert table.index.equals(sample_info.index), f"{name} 的样本顺序未对齐。"
    assert table.columns.is_unique, f"{name} 存在重复字段名。"

pd.testing.assert_frame_equal(chemical_aligned, chemical_values.loc[sample_info.index])
pd.testing.assert_frame_equal(sensory_aligned, sensory_values.loc[sample_info.index])
pd.testing.assert_frame_equal(forum_no_empty, forum_aligned.drop(columns=empty_forum_columns))
assert len(chemical_missing_positions) == 73
assert int(chemical_aligned.isna().sum().sum()) == 73
assert not forum_no_empty.isna().any().any()
assert not sensory_aligned.isna().any().any()
for table in [chemical_aligned, sensory_aligned, forum_no_empty]:
    assert not np.isinf(table.to_numpy()).any(), "数据中存在无穷值，请先检查。"

source_hashes_after = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
}
assert source_hashes_after == source_hashes_before, "输入附件在本步骤中发生变化，请先核查。"
print("验证通过：250 个样本已对齐；保留值未改变；原始缺失仍保留；附件未改变。")

验证通过：250 个样本已对齐；保留值未改变；原始缺失仍保留；附件未改变。


### 5.4 导出并重新读取

每次运行在 `处理结果/处理记录/结构整理` 或其可读版本目录中保存新文件，已有数据不覆盖。前四个 CSV 为主要数据表，其余为字段说明与处理记录。

导出后重新读取验证。`beer` 写成明确的一列；条码再读取时继续指定 `dtype={"beer_id":"string"}`，保持文本形式。


In [108]:
output_dir = next_output_path(output_root / "结构整理")
output_dir.mkdir(parents=True, exist_ok=False)

indexed_exports = {
    "01_样本信息.csv": sample_info,
    "02_化合物_保留缺失.csv": chemical_aligned,
    "03_专业感官_连续值.csv": sensory_aligned,
    "04_论坛_去全空列.csv": forum_no_empty,
    "05_化合物字段说明.csv": field_dictionary_checked,
    "06_感官字段说明.csv": target_schema
}
record_exports = {
    "07_化合物缺失位置.csv": chemical_missing_positions,
    "08_数值差异待核查.csv": positive_differences,
    "09_论坛全空列处理记录.csv": removed_forum_columns
}
for filename, table in indexed_exports.items():
    table.to_csv(output_dir / filename, index=True, encoding="utf-8-sig", mode="x")
for filename, table in record_exports.items():
    table.to_csv(output_dir / filename, index=False, encoding="utf-8-sig", mode="x")

metadata_restored = pd.read_csv(
    output_dir / "01_样本信息.csv", index_col="beer", dtype={"beer_id": "string"}
)
pd.testing.assert_frame_equal(metadata_restored, sample_info)
for filename, expected in list(indexed_exports.items())[1:4]:
    restored = pd.read_csv(output_dir / filename, index_col="beer", float_precision="round_trip")
    pd.testing.assert_frame_equal(restored, expected, check_exact=True)

source_hashes_after_export = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
}
assert source_hashes_after_export == source_hashes_before, "导出前后附件发生变化。"

notes = f"""# 结构整理结果说明

此目录由数据预处理.ipynb 第5步生成，仅进行结构整理，不包含建模或评分预测。

- 保留 {len(sample_info)} 个样本，所有主要数据表按 beer 对齐并排序。
- 化合物保留 {chemical_aligned.shape[1]} 个特征及 {len(chemical_missing_positions)} 个缺失值，尚未填补。
- 专业感官保留 {sensory_aligned.shape[1]} 个连续评分属性，未恢复为原始0–6分。
- 论坛保留 {forum_no_empty.shape[1]} 个属性，排除 {len(empty_forum_columns)} 个全空列，原因另有记录。
- 保留 {len(positive_differences)} 个数值差异记录，没有用参考值覆盖主表。
- 仍有 {len(no_reference_features)} 个化合物特征缺少直接浓度参照。
- 尚未执行缺失填补、统一缩放或统计异常值裁剪；不是最终无缺失数据。
- 感官标尺问题仍待核实，不使用空评分卡缓存的零分。
- 原附件仅被读取，未覆盖；数据导出后已重新读取验证。
"""
with (output_dir / "处理说明.md").open("x", encoding="utf-8") as handle:
    handle.write(notes)

print("结构整理数据已另存，重新读取检查通过。")
print("本次结果目录：", output_dir)
print("文件数：", len(list(output_dir.iterdir())))


结构整理数据已另存，重新读取检查通过。
本次结果目录： D:\黄洛霖的文件\竞赛\数模赛\MathorCup2026A\处理结果\处理记录\结构整理（第4版）
文件数： 10


**本阶段结果：** 10 个文件已保存，四张主要表的回读检查通过。此阶段保留原始化学缺失；填补和标准化各自另存，不覆盖这一版本。


## 6．固定样本划分并填补缺失

### 6.1 先划分样本

在每种风格中抽取约 70% 为训练部分，其余为验证部分，随机种子为 42。因各风格取整，实际为 177/73（70.8%/29.2%），两部分均覆盖 22 种风格。

检查两部分互斥且覆盖全部样本。验证部分不参与填补参数估计；以后若改用交叉验证，每折重新计算该折训练参数。


In [109]:
split_seed = 42
assert sample_info["tasting_category_fine"].notna().all()
train_ids = (
    sample_info.groupby("tasting_category_fine", sort=True)
    .sample(frac=0.7, random_state=split_seed)
    .index.sort_values()
)
validation_ids = sample_info.index.difference(train_ids).sort_values()

assert train_ids.is_unique and validation_ids.is_unique
assert len(train_ids.intersection(validation_ids)) == 0
assert set(train_ids).union(set(validation_ids)) == set(sample_info.index)

split_manifest = sample_info.copy()
split_manifest["数据用途"] = "验证部分"
split_manifest.loc[train_ids, "数据用途"] = "训练部分"
split_by_style = (
    split_manifest.groupby(["tasting_category_fine", "数据用途"])
    .size().unstack(fill_value=0)
)
assert (split_by_style[["训练部分", "验证部分"]] > 0).all().all(), "部分风格未同时进入两部分，请核对划分。"

print("训练样本数：", len(train_ids))
print("验证样本数：", len(validation_ids))
print("实际训练比例：", len(train_ids) / len(sample_info))
print("风格数：", len(split_by_style))
split_by_style

训练样本数： 177
验证样本数： 73
实际训练比例： 0.708
风格数： 22


数据用途,训练部分,验证部分
tasting_category_fine,,
Amber,8,4
Blond,22,9
Brett/cofermented,4,2
Brown,8,4
Brut,3,1
Christmas,6,3
Dubbel,7,3
Faro,2,1
Flanders old brown,4,1


### 6.2 用训练中位数只填补原缺失

对 9 个有缺失的化学特征，使用训练部分真实观测的中位数填补，验证部分应用同一参数。计算在主表现有表示上进行，不额外反变换。

这是基础处理规则，不保证恢复真实测量值或适合所有缺失机制。原有零值、负值、差异记录不属于填补对象；不借助感官标签估算化学值。


In [110]:
missing_features = chemical_aligned.columns[chemical_aligned.isna().any()]
training_observed = chemical_aligned.loc[train_ids, missing_features]
training_medians = training_observed.median()
assert training_medians.notna().all(), "有特征在训练部分完全缺失，无法计算中位数。"
assert np.isfinite(training_medians.to_numpy()).all()

fill_parameters = pd.DataFrame({
    "训练有效样本数": training_observed.notna().sum(),
    "训练缺失数": training_observed.isna().sum(),
    "验证缺失数": chemical_aligned.loc[validation_ids, missing_features].isna().sum(),
    "填充值_训练中位数": training_medians
}).rename_axis("特征")

chemical_filled = chemical_aligned.fillna(training_medians)
chemical_train_filled = chemical_filled.loc[train_ids].copy()
chemical_validation_filled = chemical_filled.loc[validation_ids].copy()

imputation_log = chemical_missing_positions.copy()
imputation_log["数据用途"] = imputation_log["beer"].map(split_manifest["数据用途"])
imputation_log["填充值"] = imputation_log["特征"].map(training_medians)
imputation_log["填补规则"] = "训练部分中位数，主表现有标尺"
assert imputation_log[["数据用途", "填充值"]].notna().all().all()
fill_parameters

,训练有效样本数,训练缺失数,验证缺失数,填充值_训练中位数
特征,,,,
X2.3.butanedione,172,5,4,0.289622
X3.methylbutanal,172,5,2,0.108457
ethyl.isobutyrate,176,1,0,0.044080
methylpyrazine,169,8,4,0.000273
furfural,176,1,0,0.007973
amyl.acetate,162,15,12,0.001996
phenylacetaldehyde,176,1,0,0.011790
ethyl.nicotinate,176,1,1,0.011517
aroma_hops_sum,168,9,4,1.962687


每行参数对应一个特征，记录训练有效观测数、两部分缺失数与填充值。`imputation_log` 保存全部 73 个原缺失位置的用途和填充值；填补值应与真实观测区分。


### 6.3 验证填补范围

检查新表无空值和无穷值、样本与字段顺序不变、原有观测逐项一致、每个填补值均等于训练中位数；含缺失副本仍保留 73 个空值。


In [111]:
assert chemical_filled.shape == chemical_aligned.shape
assert chemical_filled.index.equals(chemical_aligned.index)
assert chemical_filled.columns.equals(chemical_aligned.columns)
assert not chemical_filled.isna().any().any()
assert np.isfinite(chemical_filled.to_numpy()).all()

pd.testing.assert_frame_equal(
    chemical_filled.where(chemical_aligned.notna()),
    chemical_aligned,
    check_exact=True
)
pd.testing.assert_series_equal(
    training_medians,
    chemical_aligned.loc[train_ids, missing_features].median(),
    check_exact=True
)
for feature in missing_features:
    originally_missing = chemical_aligned[feature].isna()
    assert chemical_filled.loc[originally_missing, feature].eq(training_medians[feature]).all()

assert int(chemical_aligned.isna().sum().sum()) == 73
assert len(imputation_log) == 73
fill_summary = pd.DataFrame([
    {
        "数据用途": name,
        "样本数": len(ids),
        "填补前缺失数": int(chemical_aligned.loc[ids].isna().sum().sum()),
        "填补后缺失数": int(chemical_filled.loc[ids].isna().sum().sum())
    }
    for name, ids in [("训练部分", train_ids), ("验证部分", validation_ids)]
])
assert fill_summary["填补前缺失数"].sum() == len(imputation_log)
print("验证通过：只填补原始缺失位置，已有观测值未改变。")
fill_summary

验证通过：只填补原始缺失位置，已有观测值未改变。


,数据用途,样本数,填补前缺失数,填补后缺失数
0,训练部分,177,46,0
1,验证部分,73,27,0


### 6.4 另存数据与参数

在 `处理结果/处理记录/缺失填补` 或其可读版本目录中保存划分、填补后未缩放数据、9 行填补参数、73 行位置记录和说明，共 5 个文件。

“未缩放”指仍保留主表现有表示，不代表恢复了原始物理浓度。专业感官值不做评分转换。


In [112]:
fill_output_dir = next_output_path(output_root / "缺失填补")
fill_output_dir.mkdir(parents=True, exist_ok=False)

split_manifest.to_csv(fill_output_dir / "01_样本划分.csv", encoding="utf-8-sig", mode="x")
chemical_filled.to_csv(fill_output_dir / "02_化合物_中位数填补_未缩放.csv", encoding="utf-8-sig", mode="x")
fill_parameters.to_csv(fill_output_dir / "03_训练集填补参数.csv", encoding="utf-8-sig", mode="x")
imputation_log.to_csv(fill_output_dir / "04_逐单元格填补记录.csv", index=False, encoding="utf-8-sig", mode="x")

split_restored = pd.read_csv(
    fill_output_dir / "01_样本划分.csv", index_col="beer", dtype={"beer_id": "string"}
)
filled_restored = pd.read_csv(
    fill_output_dir / "02_化合物_中位数填补_未缩放.csv", index_col="beer", float_precision="round_trip"
)
pd.testing.assert_frame_equal(split_restored, split_manifest)
pd.testing.assert_frame_equal(filled_restored, chemical_filled, check_exact=True)

source_hashes_after_fill = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
}
assert source_hashes_after_fill == source_hashes_before, "附件内容发生变化，请先核查。"

fill_notes = f"""# 缺失填补结果说明

本目录由数据预处理.ipynb 第6步生成，仅进行数据划分和中位数填补，不训练预测模型。

- 按每种风格约70%抽取训练样本，随机种子为 {split_seed}。
- 实际训练样本 {len(train_ids)} 个，验证样本 {len(validation_ids)} 个，两部分风格均完整。
- 只用训练部分真实观测计算 {len(training_medians)} 个特征的中位数，验证部分不参与参数计算。
- 共填补 {len(imputation_log)} 个原始缺失单元格，全部位置与填充值有记录。
- 原本非空的数值，包括零、负值和待核查差异，均未修改。
- 填补在主表现有标尺上完成，尚未统一缩放或反变换浓度。
- 中位数填补是简单基线，不等于恢复真实测量；缺失机制仍需结合来源核查。
- 81个差异及161个缺少直接浓度参照的特征仍保留原有状态。
- 本目录不包含新感官评分；原始0–6分标尺问题未在此解决。
- 若后续采用交叉验证，每折必须重新计算该折训练部分的填补参数，不能直接复用本目录的固定留出参数。
- 第5步结构整理结果仍保留于：{output_dir}
- 附件和已有结果未覆盖；填补后化合物表已重新读取验证。
"""
with (fill_output_dir / "处理说明.md").open("x", encoding="utf-8") as handle:
    handle.write(fill_notes)

print("缺失填补结果已另存，重新读取检查通过。")
print("本次填补结果目录：", fill_output_dir)
print("化合物表剩余缺失数：", int(chemical_filled.isna().sum().sum()))
print("文件数：", len(list(fill_output_dir.iterdir())))


缺失填补结果已另存，重新读取检查通过。
本次填补结果目录： D:\黄洛霖的文件\竞赛\数模赛\MathorCup2026A\处理结果\处理记录\缺失填补（第3版）
化合物表剩余缺失数： 0
文件数： 5


**本阶段结果：** 训练部分填补 46 个位置，验证部分填补 27 个位置，合计 73 个；填补后化学表为 250×231，无空值，已有观测不变。导出回读检查通过。


## 7．生成化学特征标准化副本

### 7.1 只用训练部分估计参数

在填补后训练数据上计算均值和总体标准差 `ddof=0`，公式为 `z=(x−训练均值)/训练标准差`。验证部分使用相同参数。

若训练列恒定，记录标记并将实际除数设为 1，保留列与验证部分的真实变化。本次训练常量列为 0。未缩放数据仍保留，感官标签不再标准化。


In [113]:
scaling_mean = chemical_train_filled.mean()
scaling_std = chemical_train_filled.std(ddof=0)
constant_in_training = chemical_train_filled.nunique().le(1)
scaling_scale = scaling_std.mask(constant_in_training, 1.0)

assert np.isfinite(scaling_mean.to_numpy()).all()
assert np.isfinite(scaling_scale.to_numpy()).all()
assert scaling_scale.gt(0).all(), "存在无法使用的标准差，请先检查。"

scaling_parameters = pd.DataFrame({
    "训练均值": scaling_mean,
    "训练标准差_ddof0": scaling_std,
    "实际除数": scaling_scale,
    "训练常量列": constant_in_training
}).rename_axis("特征")

assert len(scaling_parameters) == chemical_filled.shape[1]
print("标准化参数来自的训练样本数：", len(chemical_train_filled))
print("特征数：", len(scaling_parameters))
print("训练常量列数：", int(constant_in_training.sum()))
scaling_parameters.head(10)

标准化参数来自的训练样本数： 177
特征数： 231
训练常量列数： 0


,训练均值,训练标准差_ddof0,实际除数,训练常量列
特征,,,,
acetaldehyde,0.588900,0.343281,0.343281,False
CS2,-1.887702,0.574087,0.574087,False
DES,-0.336150,0.624699,0.624699,False
DMS,1.391034,0.345697,0.345697,False
ethyl_2.methyl_butyrate,-2.394067,0.632555,0.632555,False
ethyl_acetate,1.397105,0.333769,0.333769,False
ethyl_decanoate,-1.032875,0.500229,0.500229,False
ethyl_hexanoate,-0.682578,0.334506,0.334506,False
ethyl_isovalerate,-2.863522,1.241551,1.241551,False


### 7.2 应用同一组参数

生成新表 `chemical_standardized`，不修改未缩放副本。已取对数的特征在现有表示上缩放，不再次取对数。

验证部分均值无需等于 0、标准差无需等于 1；标准化不保证正态性，也不恢复物理浓度或自动消除极端值。


In [114]:
chemical_standardized = chemical_filled.sub(scaling_mean, axis="columns").div(scaling_scale, axis="columns")
chemical_train_standardized = chemical_standardized.loc[train_ids].copy()
chemical_validation_standardized = chemical_standardized.loc[validation_ids].copy()

standardization_audit = pd.DataFrame({
    "训练均值_标准化后": chemical_train_standardized.mean(),
    "训练标准差_标准化后_ddof0": chemical_train_standardized.std(ddof=0),
    "验证均值_标准化后": chemical_validation_standardized.mean(),
    "验证标准差_标准化后_ddof0": chemical_validation_standardized.std(ddof=0),
    "验证最大绝对z": chemical_validation_standardized.abs().max(),
    "训练常量列": constant_in_training
}).rename_axis("特征")

print("标准化后化合物表形状：", chemical_standardized.shape)
print("验证部分最大绝对标准化值：", float(chemical_validation_standardized.abs().max().max()))
standardization_audit.sort_values("验证最大绝对z", ascending=False).head(10)

标准化后化合物表形状： (250, 231)
验证部分最大绝对标准化值： 515.8140471725497


,训练均值_标准化后,训练标准差_标准化后_ddof0,验证均值_标准化后,验证标准差_标准化后_ddof0,验证最大绝对z,训练常量列
特征,,,,,,
X.E..beta.damascenone,5.017957e-18,1.0,7.060353,59.960536,515.814047,False
X.E..2.nonenal,0.000000e+00,1.0,0.162807,1.912555,15.960306,False
citronellol,0.000000e+00,1.0,0.091312,1.637266,12.052780,False
X2.3.butanedione,1.254489e-17,1.0,0.086833,1.468513,10.123632,False
phenylacetaldehyde,-7.025140e-17,1.0,-0.044069,1.131297,8.600139,False
X3Z.hexenol,-7.025140e-17,1.0,0.073733,1.170747,8.368519,False
X2.ethyl.3.methylpyrazine,5.519753e-17,1.0,0.016672,1.118996,8.361570,False
ethyl.butyrate,4.516161e-17,1.0,0.009429,1.087546,8.204983,False
n.hexanol,-5.017957e-18,1.0,0.031159,1.066295,8.065247,False


**极端值记录：** 验证部分最大绝对 z 约为 515.814，对应 `X.E..beta.damascenone`。验证值远离训练均值或训练标准差较小都可能导致较大 z。

当前只记录，不据此认定错误，也不自动裁剪或删样本。只在训练部分检查均值约 0、非常量列标准差约 1。


### 7.3 验证数值性质与可逆性

检查标准化结果有限、结构不变；训练均值和标准差满足 10⁻¹⁰ 精度内的要求；通过 `x=z×实际除数+训练均值` 还原到填补后未缩放数据。

这里的还原是撤销标准化，不是恢复原始物理浓度。


In [115]:
assert chemical_standardized.shape == chemical_filled.shape
assert chemical_standardized.index.equals(chemical_filled.index)
assert chemical_standardized.columns.equals(chemical_filled.columns)
assert not chemical_standardized.isna().any().any()
assert np.isfinite(chemical_standardized.to_numpy()).all()

np.testing.assert_allclose(
    chemical_train_standardized.mean().to_numpy(), 0.0,
    rtol=0, atol=1e-10
)
np.testing.assert_allclose(
    chemical_train_standardized.std(ddof=0).loc[~constant_in_training].to_numpy(), 1.0,
    rtol=0, atol=1e-10
)
reconstructed_filled = chemical_standardized.mul(scaling_scale, axis="columns").add(scaling_mean, axis="columns")
np.testing.assert_allclose(
    reconstructed_filled.to_numpy(), chemical_filled.to_numpy(),
    rtol=1e-10, atol=1e-10
)
pd.testing.assert_frame_equal(chemical_filled, filled_restored, check_exact=True)

print("验证通过：标准化结果有限、结构不变，并可还原到未缩放填补数据。")
print("训练均值的最大绝对值：", float(chemical_train_standardized.mean().abs().max()))
print("非常量列训练标准差与1的最大差：", float((chemical_train_standardized.std(ddof=0).loc[~constant_in_training] - 1).abs().max()))

验证通过：标准化结果有限、结构不变，并可还原到未缩放填补数据。
训练均值的最大绝对值： 3.7735037944321704e-15
非常量列训练标准差与1的最大差： 4.218847493575595e-15


### 7.4 保存标准化数据与参数

在 `处理结果/处理记录/标准化` 或其可读版本目录中保存标准化数据、训练参数、划分及核验记录，共 5 个文件；重新读取验证数值一致。

新样本应使用已有训练参数；交叉验证时应在各折训练部分重新拟合填补和缩放参数。


In [116]:
standard_output_dir = next_output_path(output_root / "标准化")
standard_output_dir.mkdir(parents=True, exist_ok=False)

split_manifest.to_csv(standard_output_dir / "01_样本划分.csv", encoding="utf-8-sig", mode="x")
chemical_standardized.to_csv(standard_output_dir / "02_化合物_标准化.csv", encoding="utf-8-sig", mode="x")
scaling_parameters.to_csv(standard_output_dir / "03_训练集标准化参数.csv", encoding="utf-8-sig", mode="x")
standardization_audit.to_csv(standard_output_dir / "04_标准化核验记录.csv", encoding="utf-8-sig", mode="x")

standardized_restored = pd.read_csv(
    standard_output_dir / "02_化合物_标准化.csv", index_col="beer", float_precision="round_trip"
)
scaling_restored = pd.read_csv(
    standard_output_dir / "03_训练集标准化参数.csv", index_col="特征", float_precision="round_trip"
)
pd.testing.assert_frame_equal(standardized_restored, chemical_standardized, check_exact=True)
pd.testing.assert_frame_equal(scaling_restored, scaling_parameters, check_exact=True)
assert {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
} == source_hashes_before, "附件发生变化，请先核查。"

standard_notes = f"""# 标准化结果说明

本目录由数据预处理.ipynb 第7步生成，仅做化合物特征标准化，不训练模型或推断感官分数。

- 样本共 {len(chemical_standardized)} 个，化合物特征共 {chemical_standardized.shape[1]} 个。
- 沿用固定的训练/验证划分：{len(train_ids)} / {len(validation_ids)}，随机种子 {split_seed}。
- 仅用填补后的训练部分计算均值及总体标准差（ddof=0），验证部分不参与参数计算。
- 标准化公式：z = (x - 训练均值) / 实际除数。
- 训练常量列共 {int(constant_in_training.sum())} 列；如有常量列，实际除数设为1并记录。
- 标准化后无空值或无穷值，训练数值性质、反标准化及导出再读取检查均通过。
- 验证部分最大绝对z约为 {float(chemical_validation_standardized.abs().max().max()):.6f}，仅记录，未自动裁剪或删样本。
- 标准化不恢复物理浓度，不改变感官连续值，也不解决原始0–6分标尺。
- 81个差异和161个缺少直接浓度参照的特征仍保留原有处理状态。
- 未缩放填补数据和填补参数继续保留于：{fill_output_dir}
- 若以后进行交叉验证，每折重新计算填补和缩放参数，不能复用本固定留出划分的参数来评估全部折。
- 附件和既有结果未覆盖。
"""
with (standard_output_dir / "处理说明.md").open("x", encoding="utf-8") as handle:
    handle.write(standard_notes)

print("标准化数据已另存，重新读取检查通过。")
print("本次标准化结果目录：", standard_output_dir)
print("文件数：", len(list(standard_output_dir.iterdir())))


标准化数据已另存，重新读取检查通过。
本次标准化结果目录： D:\黄洛霖的文件\竞赛\数模赛\MathorCup2026A\处理结果\处理记录\标准化（第2版）
文件数： 5


**本阶段结果：** 标准化化学表为 250×231，无空值或无穷值，训练数值性质、逆标准化及导出回读检查均通过。含缺失、未缩放填补、标准化三种副本均保留。


## 8．核验相关矩阵并汇总最终文件

### 8.1 检查矩阵格式

附件 2 和 5 应为对称方阵，对角线约为 1；附件 6 是化学指标×感官属性的交叉矩阵，不要求对称或对角线为 1。所有矩阵均检查名称唯一、数值有限、系数在 [-1,1] 内。

**实际尺寸：** 231×231、50×50、44×50。附件 6 与题面 46 行不同，保留实际 44 行，不补造指标。


In [117]:
correlation_matrices = {
    name: pd.read_csv(folder / name, index_col=0, float_precision="round_trip")
    for name in ["2.csv", "5.csv", "6.csv"]
}
matrix_format_records = []
for filename, matrix in correlation_matrices.items():
    assert matrix.index.is_unique and matrix.columns.is_unique, f"{filename} 行列名有重复。"
    assert matrix.index.notna().all() and matrix.columns.notna().all()
    values = matrix.to_numpy()
    assert np.isfinite(values).all(), f"{filename} 存在缺失或无穷值。"
    assert (np.abs(values) <= 1 + 1e-12).all(), f"{filename} 超出相关系数范围。"
    is_square = filename in ["2.csv", "5.csv"]
    if is_square:
        assert matrix.index.equals(matrix.columns), f"{filename} 两轴名称或顺序不同。"
        np.testing.assert_allclose(values, values.T, rtol=0, atol=1e-12)
        np.testing.assert_allclose(np.diag(values), 1, rtol=0, atol=1e-12)
    matrix_format_records.append({
        "附件": filename, "行数": matrix.shape[0], "列数": matrix.shape[1],
        "最小系数": float(values.min()), "最大系数": float(values.max()),
        "对称与对角线检查": "通过" if is_square else "不适用：交叉矩阵"
    })
matrix_format_audit = pd.DataFrame(matrix_format_records)
matrix_format_audit


,附件,行数,列数,最小系数,最大系数,对称与对角线检查
0,2.csv,231,231,-0.663595,1.000000,通过
1,5.csv,50,50,-0.502080,1.000000,通过
2,6.csv,44,50,-0.501235,0.817147,不适用：交叉矩阵


### 8.2 统一矩阵字段名称

利用附件 8，将矩阵显示名对应到样本表字段；只规范名称中的换行、空格和已识别的 α/β 编码差异，保存原名、规范名和对应字段。

检查所有名称可对应、对应后唯一、行列无丢失；按样本表字段顺序重排，系数逐项保持不变。最终名称记录为 656 行。


In [118]:
def normalize_matrix_label(label):
    assert isinstance(label, str) and label.strip(), "矩阵或参考表存在空名称。"
    label = label.replace("_x000D_", " ").replace("\\r", " ").replace("\\n", " ")
    label = label.replace("\r", " ").replace("\n", " ")
    return " ".join(label.replace("Î±", "α").replace("Î²", "β").split())

chemical_label_table = reference.reset_index()[["chemical_name", "field_name_R"]].dropna()
sensory_label_table = trained_names.reset_index()[["CleanName_PanelFeedback", "R_name"]].dropna()
label_maps = []
for label_table in [chemical_label_table, sensory_label_table]:
    normalized = label_table.iloc[:, 0].map(normalize_matrix_label)
    assert normalized.is_unique and label_table.iloc[:, 1].is_unique, "名称对应不是一对一。"
    label_maps.append(dict(zip(normalized, label_table.iloc[:, 1])))
chemical_label_map, sensory_label_map = label_maps

aligned_matrices = {}
matrix_name_records = []
for filename, matrix in correlation_matrices.items():
    row_map = sensory_label_map if filename == "5.csv" else chemical_label_map
    column_map = chemical_label_map if filename == "2.csv" else sensory_label_map
    mapped_axes = []
    for axis, labels, lookup in [("行", matrix.index, row_map), ("列", matrix.columns, column_map)]:
        normalized = labels.map(normalize_matrix_label)
        unknown = set(normalized).difference(lookup)
        assert not unknown, f"{filename} {axis}存在未对应名称：{unknown}"
        fields = pd.Index([lookup[label] for label in normalized])
        assert fields.is_unique, f"{filename} {axis}对应后存在重复字段。"
        mapped_axes.append(fields)
        matrix_name_records.extend(
            {"附件": filename, "轴": axis, "原名称": old, "规范名称": clean, "对应字段": field}
            for old, clean, field in zip(labels, normalized, fields)
        )
    aligned = matrix.copy()
    aligned.index, aligned.columns = mapped_axes
    row_fields = sensory_aligned.columns if filename == "5.csv" else chemical_aligned.columns
    column_fields = chemical_aligned.columns if filename == "2.csv" else sensory_aligned.columns
    assert set(aligned.index).issubset(row_fields) and set(aligned.columns).issubset(column_fields)
    if filename != "6.csv":
        assert set(aligned.index) == set(row_fields) and set(aligned.columns) == set(column_fields)
    aligned = aligned.loc[row_fields[row_fields.isin(aligned.index)], column_fields[column_fields.isin(aligned.columns)]]
    aligned.index.name = "R_name" if filename == "5.csv" else "field_name_R"
    aligned.columns.name = None
    np.testing.assert_array_equal(aligned.loc[mapped_axes[0], mapped_axes[1]].to_numpy(), matrix.to_numpy())
    aligned_matrices[filename] = aligned

matrix_name_audit = pd.DataFrame(matrix_name_records)
assert len(matrix_name_audit) == sum(sum(m.shape) for m in correlation_matrices.values())
print("三个矩阵全部对应，系数保持不变；名称记录数：", len(matrix_name_audit))
matrix_name_audit.head(10)


三个矩阵全部对应，系数保持不变；名称记录数： 656


,附件,轴,原名称,规范名称,对应字段
0,2.csv,行,Acetaldehyde,Acetaldehyde,acetaldehyde
1,2.csv,行,Carbon disulfide,Carbon disulfide,CS2
2,2.csv,行,Diethyl sulfide,Diethyl sulfide,DES
3,2.csv,行,Dimethyl sulfide,Dimethyl sulfide,DMS
4,2.csv,行,Ethyl 2-methylbutyrate,Ethyl 2-methylbutyrate,ethyl_2.methyl_butyrate
5,2.csv,行,Ethyl acetate,Ethyl acetate,ethyl_acetate
6,2.csv,行,Ethyl decanoate,Ethyl decanoate,ethyl_decanoate
7,2.csv,行,Ethyl hexanoate,Ethyl hexanoate,ethyl_hexanoate
8,2.csv,行,Ethyl 3-methylbutyrate,Ethyl 3-methylbutyrate,ethyl_isovalerate
9,2.csv,行,Ethyl octanoate,Ethyl octanoate,ethyl_octanoate


### 8.3 核对相关系数来源

题面对附件 2 写 Pearson，而 `数据说明.txt` 将附件 2、5、6 标为 Spearman。使用填补前的原始观测分别重算，只用于识别附件系数来源，不替换原系数。

缺失按每对变量共同非缺失的样本处理，方阵比较上三角，交叉矩阵比较全部系数。若用于以后预测验证中的变量选择或结构学习，相关性应由相应训练部分重新计算。


In [119]:
correlation_method_records = []
for filename, matrix in aligned_matrices.items():
    if filename == "2.csv":
        observations = chemical_aligned
    elif filename == "5.csv":
        observations = sensory_aligned
    else:
        observations = pd.concat([
            chemical_aligned.loc[:, matrix.index], sensory_aligned.loc[:, matrix.columns]
        ], axis=1)
        assert observations.columns.is_unique
    compare_mask = np.ones(matrix.shape, dtype=bool)
    if filename != "6.csv":
        compare_mask = np.triu(compare_mask, k=1)
    for method in ["pearson", "spearman"]:
        recalculated = observations.corr(method=method).loc[matrix.index, matrix.columns]
        differences = np.abs(recalculated.to_numpy() - matrix.to_numpy())[compare_mask]
        assert np.isfinite(differences).all()
        correlation_method_records.append({
            "附件": filename, "重算方法": method, "比较系数数": len(differences),
            "最大绝对差": float(differences.max()),
            "全部吻合_容差1e-6": bool((differences <= 1e-6).all())
        })
correlation_method_audit = pd.DataFrame(correlation_method_records)
matched_methods = correlation_method_audit.loc[correlation_method_audit["全部吻合_容差1e-6"]]
assert matched_methods.groupby("附件").size().reindex(aligned_matrices, fill_value=0).eq(1).all(), "来源算法未唯一核实，请先检查。"
assert matched_methods["重算方法"].eq("spearman").all(), "本次附件系数来源与预期不同，请核查。"
correlation_method_audit


,附件,重算方法,比较系数数,最大绝对差,全部吻合_容差1e-6
0,2.csv,pearson,26565,1.123426e+00,False
1,2.csv,spearman,26565,6.661338e-16,True
2,5.csv,pearson,1225,5.406450e-01,False
3,5.csv,spearman,1225,5.551115e-16,True
4,6.csv,pearson,2200,4.618016e-01,False
5,6.csv,spearman,2200,5.551115e-16,True


**核验结果：** 三个附件均与 Spearman 重算吻合，最大绝对差不超过 6.67×10⁻¹⁶；Pearson 不吻合。记录题面与实际文件的出入，保留附件系数。

### 8.4 汇总处理验收与评分标尺状态

检查对齐、划分、填补范围、训练参数、数值有限性、标准化可逆性、连续感官值和附件内容保持不变。最终 10 项验收全部通过。

`target_schema_final` 明确记录：现有评分卡输入为空，且没有其他原始评分或转换参数，无法还原原始 0–6 分。


In [120]:
target_schema_final = target_schema.copy()
target_schema_final["原始0至6观测状态"] = "现有评分卡输入为空；用户确认无其他原始评分或转换参数，无法还原"

preprocessing_checks = {
    "250个样本及各表索引一致": len(sample_info) == 250 and all(
        table.index.equals(sample_info.index)
        for table in [chemical_aligned, chemical_filled, chemical_standardized, sensory_aligned, forum_no_empty]
    ),
    "训练验证互斥且覆盖全部样本和风格": (
        not len(train_ids.intersection(validation_ids))
        and set(train_ids).union(validation_ids) == set(sample_info.index)
        and (split_by_style[["训练部分", "验证部分"]] > 0).all().all()
    ),
    "73个缺失填补且已有观测不变": (
        len(imputation_log) == int(chemical_aligned.isna().sum().sum()) == 73
        and chemical_filled.where(chemical_aligned.notna()).equals(chemical_aligned)
        and not chemical_filled.isna().any().any()
        and all(chemical_filled.loc[chemical_aligned[f].isna(), f].eq(training_medians[f]).all() for f in missing_features)
    ),
    "中位数参数仅来自训练部分": training_medians.equals(chemical_aligned.loc[train_ids, missing_features].median()),
    "标准化参数仅来自训练部分": (
        scaling_mean.equals(chemical_filled.loc[train_ids].mean())
        and scaling_std.equals(chemical_filled.loc[train_ids].std(ddof=0))
        and scaling_scale.equals(scaling_std.mask(constant_in_training, 1.0))
    ),
    "填补标准化感官及论坛值均有限": all(
        np.isfinite(table.to_numpy()).all()
        for table in [chemical_filled, chemical_standardized, sensory_aligned, forum_no_empty]
    ),
    "标准化可逆回未缩放填补数据": np.allclose(
        chemical_standardized.mul(scaling_scale, axis="columns").add(scaling_mean, axis="columns").to_numpy(),
        chemical_filled.to_numpy(), rtol=1e-10, atol=1e-10
    ),
    "感官连续值保持原样": sensory_aligned.equals(sensory_values.loc[sample_info.index]),
    "论坛副本只排除3个全空列": len(empty_forum_columns) == 3 and forum_no_empty.equals(forum_aligned.drop(columns=empty_forum_columns)),
    "附件文件内容未改变": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in folder.iterdir() if p.is_file()
    } == source_hashes_before
}
preprocessing_audit = pd.DataFrame({
    "验收项": list(preprocessing_checks), "通过": [bool(value) for value in preprocessing_checks.values()]
})
assert preprocessing_audit["通过"].all(), preprocessing_audit.loc[~preprocessing_audit["通过"]].to_dict("records")
print("基础预处理检查通过；原始0–6分不可恢复、浓度与差异等限制仍保留。")
preprocessing_audit


基础预处理检查通过；原始0–6分不可恢复、浓度与差异等限制仍保留。


,验收项,通过
0,250个样本及各表索引一致,True
1,训练验证互斥且覆盖全部样本和风格,True
2,73个缺失填补且已有观测不变,True
3,中位数参数仅来自训练部分,True
4,标准化参数仅来自训练部分,True
5,填补标准化感官及论坛值均有限,True
6,标准化可逆回未缩放填补数据,True
7,感官连续值保持原样,True
8,论坛副本只排除3个全空列,True
9,附件文件内容未改变,True


### 8.5 保存矩阵和验收记录

在 `处理结果/处理记录/矩阵与最终验收` 或其可读版本目录中保存 3 个矩阵、格式与名称检查、来源核验、感官字段说明、验收表和文件清单，共 10 个文件。

矩阵回读检查通过。既有历史记录原样归档，记录中旧位置的对应关系见 `处理记录/目录说明.md`。


In [121]:
final_output_dir = next_output_path(output_root / "矩阵与最终验收")
final_output_dir.mkdir(parents=True, exist_ok=False)

matrix_export_names = {
    "2.csv": "01_化合物相关_字段对齐.csv",
    "5.csv": "02_感官相关_字段对齐.csv",
    "6.csv": "03_化合物感官相关_字段对齐.csv"
}
for source_name, filename in matrix_export_names.items():
    expected = aligned_matrices[source_name]
    expected.to_csv(final_output_dir / filename, encoding="utf-8-sig", mode="x")
    restored = pd.read_csv(final_output_dir / filename, index_col=0, float_precision="round_trip")
    pd.testing.assert_frame_equal(restored, expected, check_exact=True)

final_records = {
    "04_矩阵格式检查.csv": matrix_format_audit,
    "05_矩阵名称对应记录.csv": matrix_name_audit,
    "06_相关系数来源核验.csv": correlation_method_audit,
    "08_预处理验收.csv": preprocessing_audit
}
for filename, table in final_records.items():
    table.to_csv(final_output_dir / filename, index=False, encoding="utf-8-sig", mode="x")
target_schema_final.to_csv(final_output_dir / "07_感官字段说明_最终.csv", encoding="utf-8-sig", mode="x")

final_notes = f"""# 现有附件的基础预处理验收

本目录由数据预处理.ipynb第8步生成，不含赛题求解结果。

- 样本数{len(sample_info)}；化学特征{chemical_filled.shape[1]}；连续感官属性{sensory_aligned.shape[1]}；论坛保留属性{forum_no_empty.shape[1]}。
- 固定划分为训练{len(train_ids)} / 验证{len(validation_ids)}，随机种子{split_seed}；填补与标准化参数仅从训练部分计算。
- 原始73个化合物缺失已用训练中位数填补，已有观测值保持不变；保留未缩放与标准化两个副本。
- 2.csv、5.csv、6.csv的实际尺寸依次为231×231、50×50、44×50；6.csv与题面46行的描述不同，未补造行。
- 三个附件的系数均与原始观测的Spearman重算结果吻合，题面对附件2写Pearson的表述与此不同。
- 矩阵仅对应名称并重排行列，系数未改；按变量对共同非缺失样本核验，无填补后重算替换。
- 全样本相关矩阵只作为附件汇总数据；以后若用于预测验证中的变量选择或结构学习，应从相应训练部分重新计算。
- 07_感官字段说明_最终.csv更新了标尺状态；旧的结构整理目录是历史版本，保留不覆盖。

## 不能由现有材料解决的限制

1. 用户确认无其他原始评分或转换参数，现有评分卡输入为空；不能还原原始0–6分，不能把连续值线性拉伸、四舍五入或把空卡缓存零分当真实分数。
2. A_有15项、F_有15项，另有20项其他感官/总体属性；全部50项保留，题面“约40项”的具体目标范围未在预处理阶段强行确定。
3. {len(no_reference_features)}个化学特征缺少直接浓度参照，不能统一宣称已恢复物理浓度。
4. {len(positive_differences)}个值超出此前绝对差容差，不代表全是错误；含微小数值差异，未用参考值覆盖。
5. 验证部分最大绝对z约{float(chemical_validation_standardized.abs().max().max()):.6f}，最大者为{chemical_validation_standardized.abs().max().idxmax()}；未自动裁剪或删样本。

## 结果定位

- 含原始缺失、连续感官、论坛及字段说明：{output_dir}
- 填补后的未缩放数据、划分及填补参数：{fill_output_dir}
- 标准化数据、划分及标准化参数：{standard_output_dir}
- 矩阵与最终验收：{final_output_dir}
- 09_结果文件清单.csv列出上述4个目录的文件，不含清单自身；其他历史运行目录保持原样。

验收通过后，现有数据的基础预处理结束；这不表示已完成原始等级评分或其他赛题任务。
"""
with (final_output_dir / "处理说明.md").open("x", encoding="utf-8") as handle:
    handle.write(final_notes)

file_manifest = pd.DataFrame([
    {"阶段": stage, "相对项目路径": str(p.relative_to(folder.parent)), "文件字节数": p.stat().st_size}
    for stage, directory in [
        ("结构整理", output_dir), ("缺失填补", fill_output_dir),
        ("标准化", standard_output_dir), ("矩阵与最终验收", final_output_dir)
    ]
    for p in sorted(directory.iterdir()) if p.is_file()
])
file_manifest.to_csv(final_output_dir / "09_结果文件清单.csv", index=False, encoding="utf-8-sig", mode="x")
assert len(list(final_output_dir.iterdir())) == 10
assert {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
} == source_hashes_before, "附件发生变化，请核查。"
print("基础预处理最终验收通过，矩阵回读检查通过。")
print("本次最终结果目录：", final_output_dir)
print("最终目录文件数：", len(list(final_output_dir.iterdir())))
file_manifest


基础预处理最终验收通过，矩阵回读检查通过。
本次最终结果目录： D:\黄洛霖的文件\竞赛\数模赛\MathorCup2026A\处理结果\处理记录\矩阵与最终验收
最终目录文件数： 10


,阶段,相对项目路径,文件字节数
0,结构整理,处理结果\处理记录\结构整理（第4版）\01_样本信息.csv,6644
1,结构整理,处理结果\处理记录\结构整理（第4版）\02_化合物_保留缺失.csv,694163
2,结构整理,处理结果\处理记录\结构整理（第4版）\03_专业感官_连续值.csv,157850
3,结构整理,处理结果\处理记录\结构整理（第4版）\04_论坛_去全空列.csv,581479
4,结构整理,处理结果\处理记录\结构整理（第4版）\05_化合物字段说明.csv,23156
5,结构整理,处理结果\处理记录\结构整理（第4版）\06_感官字段说明.csv,8305
6,结构整理,处理结果\处理记录\结构整理（第4版）\07_化合物缺失位置.csv,1423
7,结构整理,处理结果\处理记录\结构整理（第4版）\08_数值差异待核查.csv,6490
8,结构整理,处理结果\处理记录\结构整理（第4版）\09_论坛全空列处理记录.csv,331
9,结构整理,处理结果\处理记录\结构整理（第4版）\处理说明.md,829


### 8.6 汇总编号 1–8 的最终文件

最终数据单独放在 `处理结果/最终数据`，该目录只包含下面 8 个文件。重复导出时使用可读的版本目录，旧文件保留。

| 编号 | 文件内容 | 本次处理 |
| --- | --- | --- |
| 1 | 化合物与样本信息 | 原缺失已填补，化学特征已标准化 |
| 2 | 化合物相关矩阵 | 字段对应与行列重排，原系数保留 |
| 3 | 感官评分卡 | 原样复制，仍为空白评分模板 |
| 4 | 专业感官与样本信息 | 按 beer 对齐，连续值不变 |
| 5 | 感官相关矩阵 | 字段对应与行列重排，原系数保留 |
| 6 | 化合物与感官交叉矩阵 | 字段对应，保留实际 44×50 |
| 7 | 论坛属性与样本信息 | 按 beer 对齐，排除 3 个全空列 |
| 8 | 综合参考数据 | 原样复制，工作表与内容不变 |

1、4、7 均带 `beer`、`beer_id` 和 `tasting_category_fine`。1 号化学特征为 Z-score，不是原始物理浓度；2、5、6 保留原附件的 Spearman 汇总系数，不声称等于填补后的重算结果。参数、未缩放数据和检查记录保存在 `处理记录` 中。


In [ ]:
from shutil import copyfileobj

final_data_dir = next_output_path(output_root.parent / "最终数据")
final_data_dir.mkdir(parents=True, exist_ok=False)

final_sample_tables = {
    "1_化合物数据_已填补标准化.csv": sample_info.join(chemical_standardized, validate="one_to_one"),
    "4_专业感官数据.csv": sample_info.join(sensory_aligned, validate="one_to_one"),
    "7_论坛感官数据.csv": sample_info.join(forum_no_empty, validate="one_to_one")
}
for filename, table in final_sample_tables.items():
    assert table.index.equals(sample_info.index) and table.columns.is_unique
    assert not table.isna().any().any()
    table.to_csv(final_data_dir / filename, encoding="utf-8-sig", mode="x")
    restored = pd.read_csv(
        final_data_dir / filename, index_col="beer", dtype={"beer_id": "string"}, float_precision="round_trip"
    )
    pd.testing.assert_frame_equal(restored, table, check_exact=True)

final_matrix_names = {
    "2.csv": "2_化合物相关矩阵.csv",
    "5.csv": "5_感官相关矩阵.csv",
    "6.csv": "6_化合物与感官相关矩阵.csv"
}
for source_name, filename in final_matrix_names.items():
    expected = aligned_matrices[source_name]
    expected.to_csv(final_data_dir / filename, encoding="utf-8-sig", mode="x")
    restored = pd.read_csv(final_data_dir / filename, index_col=0, float_precision="round_trip")
    pd.testing.assert_frame_equal(restored, expected, check_exact=True)

for source_name, filename in [("3.xlsx", "3_感官评分卡.xlsx"), ("8.xlsx", "8_参考数据.xlsx")]:
    destination = final_data_dir / filename
    with (folder / source_name).open("rb") as original, destination.open("xb") as copied:
        copyfileobj(original, copied)
    assert hashlib.sha256(destination.read_bytes()).hexdigest() == source_hashes_before[source_name]

assert len(list(final_data_dir.iterdir())) == 8
assert {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in folder.iterdir() if p.is_file()
} == source_hashes_before, "附件发生变化，请核查。"
final_file_list = pd.DataFrame([
    {"附件编号": int(p.name.split("_", 1)[0]), "最终文件": p.name}
    for p in sorted(final_data_dir.iterdir())
])
print("编号1–8的最终文件已整理，导出回读与原样副本检查通过。")
print("最终数据目录：", final_data_dir)
final_file_list


**完成状态：** 现有数据的基础预处理与编号 1–8 的最终文件整理已完成。复现时运行全部代码；已有输出不覆盖，会生成清楚的版本编号。

**仍需保留的限制：** 原始 0–6 分不可恢复；A_/F_ 实际共 30 项与题面“约 40 项”不一致；161 个特征缺少直接浓度参照；81 个差异记录尚不能认定为错误；验证部分较大 z 值未自动裁剪。

附件 2、5、6 仍为填补前全样本观测的相关汇总，不是对最终填补后数据重新计算的矩阵。评分卡与参考工作簿只是原样副本。本次未开展赛题求解。
